#Scrapping from Play Store

In [3]:
import warnings
warnings.filterwarnings(action='ignore')

!pip install google-play-scraper demoji nltk wordcloud matplotlib Levenshtein
!pip install --upgrade git+https://github.com/ariaghora/mpstemmer.git


import json
import re
import string
import demoji
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from google_play_scraper import Sort, reviews
from mpstemmer import MPStemmer
from nltk.corpus import stopwords

nltk.download('stopwords')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.1 MB/s eta 0:00:00
  Cloning https://github.com/ariaghora/mpstemmer.git to /tmp/pip-req-build-nykuh2z3
  Running command git clone --filter=blob:none --quiet https://github.com/ariaghora/mpstemmer.git /tmp/pip-req-build-nykuh2z3
  Resolved https://github.com/ariaghora/mpstemmer.git to commit 25a5fd923af163a7eac3a5ec976984156ca8fa8b
  Preparing metadata (setup.py) ... done
  Created wheel for mpstemmer: filename=mpstemmer-0.1.0-py3-none-any.whl size=99832 sha256=a4fd869d023162443e18b73675f249ac17582b2cff2e86a2c3f66ff1d71ced77
  Stored in directory: /tmp/pip-ephem-wheel-cache-3cxbfewa/wheels/d0/b9/fd/dba0c7d44e5338db51da6ffcbe96db4326db3996a403f5ced1
Successfully built mpstemmer


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
# Scraping ulasan Mobile Legends dari Play Store
result, _ = reviews(
    'com.mobile.legends',
    lang='en',
    country='Us',
    sort=Sort.NEWEST,
    count=100
)

with open("response.json", "w", encoding='utf-8') as f:
    json.dump(result, f, indent=4, ensure_ascii=False, default=str)

In [5]:
import json
import pandas as pd
from langdetect import detect


with open("response.json", "r", encoding='utf-8') as f:
    data = json.load(f)

authors = []
comments = []

for item in data:
    text = item["content"]
    try:

        if detect(text) == 'en':
            authors.append(item["userName"])
            comments.append(text)
    except:
        pass

df = pd.DataFrame({'Authors': authors, 'Comments': comments})
df.rename(columns={'Comments': 'text'}, inplace=True)
df

ModuleNotFoundError: No module named 'langdetect'

In [ ]:
df.info()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = ' '.join(df['text'].dropna().astype(str))

wordcloud = WordCloud(width=1000, height=500, background_color='white', max_words=1000).generate(text)

plt.figure(figsize=(7, 4))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
import re
import string
import demoji
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer # Mengganti MPStemmer dengan PorterStemmer

nltk.download('stopwords')

In [ ]:
# ini fungsinya untuk mengRemove HTML ya (khan,rin)
def remove_html(text):
    html_pattern = re.compile('<.*?>')
    return html_pattern.sub(r'', text)

In [ ]:
#  menghapus Hashtag
def remove_hashtag(text):
    return re.sub(r'#\w+', '', text)

In [ ]:
 #Remove Links & Email
def remove_rls(text):
    text = re.sub(r'https?://\S+', '', str(text))
    text = re.sub(r'http\S+', '', str(text))
    text = re.sub(r'www\S+', '', str(text))
    text = re.sub(r'\S+@\S+', '', str(text))
    return text

In [ ]:
#  Remove Punctuation(ngehapus simbol simbol dan emotikon yang pake simbol ) List & Emoticon Teks
def remove_punctuation_list(text):
    text = re.sub(r'(:\)|:-\)|:D|:-D|;\)|;-\))', '', text)
    text = re.sub(r'(:\(|:-\()', '', text)
    text = re.sub(r'(:v|:-v)', '', text)
    text = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
def remove_emoji(text):
    cleaned_text = demoji.replace(text, repl="")
    cleaned_text = re.sub(r'[^\x00-\x7F]+', '', cleaned_text)
    return cleaned_text

In [ ]:
# Inisialisasi PorterStemmer untuk Bahasa Inggris
ps = PorterStemmer()


In [ ]:
# Stopwords Bahasa Inggris
sw_english = set(stopwords.words('english')) - {'not', 'no', 'nor', 'neither'}

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in sw_english])

In [ ]:
df['rm_html'] = df['text'].apply(remove_html)
df.head(20)

In [ ]:
df['rm_hashtag'] = df['rm_html'].apply(remove_hashtag)
df[['rm_html','rm_hashtag']].head(20)

In [ ]:
df['lower_case'] = df['rm_hashtag'].str.lower()
df[['rm_hashtag', 'lower_case']].head(20)

In [ ]:
df['rm_url'] = df['lower_case'].apply(remove_rls)
df[['lower_case','rm_url']].head(20)

In [ ]:
df['rm_punc'] = df['rm_url'].apply(remove_punctuation_list)
df[['rm_url','rm_punc']].head(20)

In [ ]:
df['rm_stopwords'] = df['rm_punc'].apply(lambda x: " ".join([ps.stem(word) for word in str(x).split()]) if isinstance(x, str) else '')
df[['rm_punc','rm_stopwords']].head(20)


In [ ]:
df['rm_stopwords'] = df['stem'].apply(remove_stopwords)
df[['stem','rm_stopwords']].head(20)

In [ ]:



df['rm_punc'] = df['rm_url'].apply(remove_punctuation_list)

df['rm_stopwords'] = df['rm_punc'].apply(remove_stopwords)

df['rm_emoji'] = df['rm_stopwords'].apply(remove_emoji)

df[['rm_stopwords', 'rm_emoji']].head(20)




In [ ]:
text = ' '.join(df['rm_stopwords'].dropna().astype(str))

wordcloud = WordCloud(width=1000, height=500, background_color='white', max_words=100).generate(text)

plt.figure(figsize=(7, 4))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
df.rename(columns={'rm_stopwords': 'prepro'}, inplace=True)
df.info()

# Menyimpan hasil akhir ke file CSV (Sama persis seperti contoh dosen)
df[['Authors','text','prepro']].to_csv('mobile_legends_preprocessed.csv', index=False)

In [8]:
# 1. Install dependensi yang kurang agar pemrosesan data bisa berjalan
!pip install langdetect

# 2. Re-create DataFrame dari file hasil scraping response.json
import json
import pandas as pd
from langdetect import detect

with open("response.json", "r", encoding='utf-8') as f:
    data = json.load(f)

authors = []
comments = []
for item in data:
    text = item["content"]
    try:
        if detect(text) == 'en':
            authors.append(item["userName"])
            comments.append(text)
    except:
        pass

df = pd.DataFrame({'Authors': authors, 'text': comments})

# Jalankan fungsi preprocessing minimal untuk menghasilkan kolom 'prepro'
# karena cell-cell preprocessing Anda bergantung pada df ini
from nltk.corpus import stopwords
import demoji
import re

sw_english = set(stopwords.words('english')) - {'not', 'no', 'nor', 'neither'}
def remove_stopwords(t):
    return " ".join([word for word in str(t).split() if word not in sw_english])

def remove_emoji(t):
    return re.sub(r'[^\x00-\x7F]+', '', demoji.replace(t, repl=""))

df['prepro'] = df['text'].str.lower().apply(remove_stopwords).apply(remove_emoji)

# 3. Simpan DataFrame tersebut menjadi file CSV di workspace Google Colab
df[['Authors', 'text', 'prepro']].to_csv('mobile_legends_preprocessed.csv', index=False)
print("File 'mobile_legends_preprocessed.csv' berhasil disimpan di workspace Colab!")

# 4. Download file CSV langsung ke komputer Anda
from google.colab import files
files.download('mobile_legends_preprocessed.csv')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993335 sha256=6a7acc2cec8635e9dea5dfdf0eb628bae2469dfa4285b1b491376ca6ca5c42a9
  Stored in directory: /root/.cache/pip/wheels/eb/87/25/2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect
File 'mobile_legends_preprocessed.csv' berhasil disimpan di workspace Colab!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Feature Engginering

In [ ]:
#Try

!pip install scikit-learn gensim pandas numpy
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec, FastText

In [ ]:
dokumen = [
    "data science itu seru dan menantang",
    "beljar data science butuh waktu dan usaha",
    "machine learning adalah bagian dari data science",
    "saya suka belajar machine learning setiap hari",
]

vocab = set()
for doc in dokumen:
  for word in doc.split():
    vocab.add(word)
    word2idx = {word: idx for idx, word in enumerate(vocab)}

def one_hot(kata):
  one_hot_kata = np.zeros(len(vocab))
  one_hot_kata[word2idx[kata]]
  return one_hot_kata

  pass

In [ ]:
vectorizer = CountVectorizer(binary=False)
X = vectorizer.fit_transform(dokumen)
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

In [9]:
!pip install gensim
import pandas as pd, numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec, FastText

# Membaca data - pastikan file 'ulasan_clean.csv' atau file hasil preprocessing Anda ada di direktori kerja
# Jika nama filennya 'mobile_legends_preprocessed.csv', Anda bisa menyesuaikannya di sini.
try:
    df = pd.read_csv("ulasan_clean.csv").dropna(subset=["ulasan_clean"])
except FileNotFoundError:
    # fallback jika filenya bernama mobile_legends_preprocessed.csv
    df = pd.read_csv("mobile_legends_preprocessed.csv").dropna(subset=["prepro"])
    df.rename(columns={"prepro": "ulasan_clean"}, inplace=True)

df["ulasan_clean"] = df["ulasan_clean"].astype(str)

# One-Hot (kata)
vocab = sorted(set(w for s in df["ulasan_clean"] for w in s.split()))
idx = {w: i for i, w in enumerate(vocab)}
def one_hot(w):
    v = np.zeros(len(vocab)); v[idx[w]] = 1
    return v

# BoW dan TF-IDF
bow = CountVectorizer(max_features=1000).fit_transform(df["ulasan_clean"])
tfidf = TfidfVectorizer(max_features=1000).fit_transform(df["ulasan_clean"])

# Word2Vec: sg=0 CBOW, sg=1 Skip-gram
tokens = [s.split() for s in df["ulasan_clean"]]
cbow = Word2Vec(tokens, vector_size=100, window=5, min_count=2, sg=0)
skip = Word2Vec(tokens, vector_size=100, window=5, min_count=2, sg=1)

# Mencari kata terdekat (menggunakan kata yang ada dalam kosakata Anda)
try:
    print(skip.wv.most_similar("game"))
except KeyError:
    # Fallback jika kata 'game' tidak ada di kosakata Anda yang minim
    if len(skip.wv) > 0:
        print(skip.wv.most_similar(skip.wv.index_to_key[0]))

# FastText: bisa membuat vektor kata yang tidak ada di vocab
ft = FastText(tokens, vector_size=100, window=5, min_count=2, sg=1)
print(ft.wv["game"][:5])

[('years,', 0.3381071984767914), ('lag', 0.2824361324310303), ('playing', 0.2629139721393585), ('making', 0.25885340571403503), ('game.', 0.23458658158779144), ('fun', 0.2213670164346695), ('horrible', 0.21551617980003357), ('really', 0.20997509360313416), ('hope', 0.19417724013328552), ('big', 0.19380593299865723)]
[ 0.0002968   0.00172687 -0.00140363 -0.00021675 -0.00031904]
